# CNN + BiLTSM

## Cargar librerías

In [ ]:
import os, random, numpy as np
from pathlib import Path
from typing import Dict, Any

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor, Callback
from pytorch_lightning.loggers import CSVLogger
from torchmetrics.classification import MulticlassAccuracy, MulticlassPrecision, MulticlassRecall, MulticlassF1Score

## CONFIG

In [ ]:
DATA_ROOT    = Path(r"C:\Users\IALAB\Documents\GitHub\data\segmentación5seg_LogMel2D\segmentación5seg_LogMel2D")   # <-- AJUSTA
SPLITS       = {"train": "train", "val": "val", "test": "test"}

IMG_SIZE     = (224, 224)
BATCH_SIZE   = 128
NUM_WORKERS  = 0 if os.name == "nt" else 4
MAX_EPOCHS   = 100
LR           = 2e-4
WD           = 1e-4
DROPOUT      = 0.3
LSTM_HIDDEN  = 256
LSTM_LAYERS  = 2
SEED         = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision("high")

## DATAMODULE

In [ ]:
class SpectrogramDataModule(pl.LightningDataModule):
    def __init__(self, data_root: Path, splits: dict, batch_size: int, num_workers: int):
        super().__init__()
        self.data_root   = Path(data_root)
        self.splits      = splits
        self.batch_size  = batch_size
        self.num_workers = num_workers
        self.tx_train = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize(IMG_SIZE, antialias=True),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5]),
        ])
        self.tx_eval = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize(IMG_SIZE, antialias=True),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5]),
        ])
    def setup(self, stage=None):
        root = self.data_root
        self.ds_train = ImageFolder(root / self.splits["train"], transform=self.tx_train)
        self.ds_val   = ImageFolder(root / self.splits["val"],   transform=self.tx_eval)
        self.ds_test  = ImageFolder(root / self.splits["test"],  transform=self.tx_eval)
        self.num_classes = len(self.ds_train.classes)
    def train_dataloader(self):
        return DataLoader(self.ds_train, batch_size=self.batch_size, shuffle=True,
                          num_workers=self.num_workers, pin_memory=True)
    def val_dataloader(self):
        return DataLoader(self.ds_val, batch_size=self.batch_size, shuffle=False,
                          num_workers=self.num_workers, pin_memory=True)
    def test_dataloader(self):
        return DataLoader(self.ds_test, batch_size=self.batch_size, shuffle=False,
                          num_workers=self.num_workers, pin_memory=True)

## MODELO

In [ ]:
class CNNEncoder(nn.Module):
    def __init__(self, in_ch=1, dropout=0.3):
        super().__init__()
        c1,c2,c3,c4 = 32,64,128,256
        self.block1 = nn.Sequential(
            nn.Conv2d(in_ch,c1,3,padding=1), nn.BatchNorm2d(c1), nn.ReLU(inplace=True),
            nn.Conv2d(c1,c1,3,padding=1),    nn.BatchNorm2d(c1), nn.ReLU(inplace=True),
            nn.MaxPool2d((2,2)), nn.Dropout(dropout)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(c1,c2,3,padding=1), nn.BatchNorm2d(c2), nn.ReLU(inplace=True),
            nn.Conv2d(c2,c2,3,padding=1), nn.BatchNorm2d(c2), nn.ReLU(inplace=True),
            nn.MaxPool2d((2,2)), nn.Dropout(dropout)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(c2,c3,3,padding=1), nn.BatchNorm2d(c3), nn.ReLU(inplace=True),
            nn.Conv2d(c3,c3,3,padding=1), nn.BatchNorm2d(c3), nn.ReLU(inplace=True),
            nn.MaxPool2d((2,2)), nn.Dropout(dropout)
        )
        self.block4 = nn.Sequential(
            nn.Conv2d(c3,c4,3,padding=1), nn.BatchNorm2d(c4), nn.ReLU(inplace=True),
            nn.Conv2d(c4,c4,3,padding=1), nn.BatchNorm2d(c4), nn.ReLU(inplace=True),
            nn.MaxPool2d((2,1)), nn.Dropout(dropout)   # pool solo en frecuencia
        )
    def forward(self, x):
        x = self.block1(x); x = self.block2(x); x = self.block3(x); x = self.block4(x)
        return x  # [B,256,F',T']

class CNNBiLSTMClassifier(pl.LightningModule):
    def __init__(self, num_classes, lr=2e-4, wd=1e-4, dropout=0.3, lstm_hidden=256, lstm_layers=2):
        super().__init__()
        self.save_hyperparameters()
        self.enc = CNNEncoder(1, dropout)
        self.freq_pool = nn.AdaptiveAvgPool2d((1, None))           # [B,256,1,T']
        self.lstm = nn.LSTM(256, lstm_hidden, num_layers=lstm_layers,
                            batch_first=True, bidirectional=True, dropout=dropout)
        self.head = nn.Sequential(
            nn.Linear(2*lstm_hidden, 2*lstm_hidden), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(2*lstm_hidden, num_classes)
        )
        self.crit = nn.CrossEntropyLoss(label_smoothing=0.1)
        # métricas
        self.train_acc  = MulticlassAccuracy(num_classes=num_classes, average='macro')
        self.val_acc    = MulticlassAccuracy(num_classes=num_classes, average='macro')
        self.test_acc   = MulticlassAccuracy(num_classes=num_classes, average='macro')
        self.train_prec = MulticlassPrecision(num_classes=num_classes, average='macro')
        self.val_prec   = MulticlassPrecision(num_classes=num_classes, average='macro')
        self.test_prec  = MulticlassPrecision(num_classes=num_classes, average='macro')
        self.train_rec  = MulticlassRecall(num_classes=num_classes, average='macro')
        self.val_rec    = MulticlassRecall(num_classes=num_classes, average='macro')
        self.test_rec   = MulticlassRecall(num_classes=num_classes, average='macro')
        self.train_f1   = MulticlassF1Score(num_classes=num_classes, average='macro')
        self.val_f1     = MulticlassF1Score(num_classes=num_classes, average='macro')
        self.test_f1    = MulticlassF1Score(num_classes=num_classes, average='macro')

    def forward(self, x):
        f = self.enc(x)                       # [B,256,F',T']
        f = self.freq_pool(f).squeeze(2)      # [B,256,T']
        f = f.permute(0,2,1)                  # [B,T',256]
        out,_ = self.lstm(f)                  # [B,T',2*h]
        h = out.mean(dim=1)                   # [B,2*h]
        return self.head(h)                   # [B,C]

    def _shared_step(self, batch, stage):
        x,y = batch
        logits = self(x)
        loss = self.crit(logits, y)
        preds = logits.argmax(1)
        if stage=='train':
            self.train_acc.update(preds,y); self.train_prec.update(preds,y)
            self.train_rec.update(preds,y); self.train_f1.update(preds,y)
            self.log('train/loss', loss, on_step=False, on_epoch=True, prog_bar=False)
        elif stage=='val':
            self.val_acc.update(preds,y); self.val_prec.update(preds,y)
            self.val_rec.update(preds,y); self.val_f1.update(preds,y)
            self.log('val/loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        else:
            self.test_acc.update(preds,y); self.test_prec.update(preds,y)
            self.test_rec.update(preds,y); self.test_f1.update(preds,y)
            self.log('test/loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):   return self._shared_step(batch,'train')
    def validation_step(self, batch, batch_idx): return self._shared_step(batch,'val')
    def test_step(self, batch, batch_idx):       return self._shared_step(batch,'test')

    def on_train_epoch_end(self):
        acc=self.train_acc.compute(); self.train_acc.reset()
        pr =self.train_prec.compute(); self.train_prec.reset()
        rc =self.train_rec.compute();  self.train_rec.reset()
        f1 =self.train_f1.compute();   self.train_f1.reset()
        self.log_dict({'train/acc':acc,'train/prec':pr,'train/rec':rc,'train/f1':f1}, prog_bar=True)

    def on_validation_epoch_end(self):
        acc=self.val_acc.compute(); self.val_acc.reset()
        pr =self.val_prec.compute(); self.val_prec.reset()
        rc =self.val_rec.compute();  self.val_rec.reset()
        f1 =self.val_f1.compute();   self.val_f1.reset()
        self.log_dict({'val/acc':acc,'val/prec':pr,'val/rec':rc,'val/f1':f1}, prog_bar=True)

    def on_test_epoch_end(self):
        acc=self.test_acc.compute(); self.test_acc.reset()
        pr =self.test_prec.compute(); self.test_prec.reset()
        rc =self.test_rec.compute();  self.test_rec.reset()
        f1 =self.test_f1.compute();   self.test_f1.reset()
        self.log_dict({'test/acc':acc,'test/prec':pr,'test/rec':rc,'test/f1':f1}, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=LR, weight_decay=WD)
        def sch_lambda(epoch):
            warmup=5
            if epoch < warmup: return (epoch+1)/warmup
            t = (epoch-warmup)/max(1,(MAX_EPOCHS-warmup))
            return 0.5*(1+np.cos(np.pi*t))
        sch = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=sch_lambda)
        return {"optimizer": opt, "lr_scheduler": sch}

## Callback

In [ ]:
class EpochMetricsPrinter(Callback):
    """Imprime métricas de train/val al cierre de cada época."""
    KEYS = [
        'train/loss','train/acc','train/prec','train/rec','train/f1',
        'val/loss','val/acc','val/prec','val/rec','val/f1'
    ]
    def _fmt(self, x: Any):
        try:
            return f"{float(x):.4f}"
        except Exception:
            return str(x)
    def on_validation_epoch_end(self, trainer: pl.Trainer, pl_module: pl.LightningModule):
        m: Dict[str, Any] = trainer.callback_metrics
        e = trainer.current_epoch
        vals = {k:self._fmt(m.get(k, float('nan'))) for k in self.KEYS}
        print(
            f"[Epoch {e:03d}] "
            f"TRAIN: loss={vals['train/loss']}  acc={vals['train/acc']}  "
            f"prec={vals['train/prec']}  rec={vals['train/rec']}  f1={vals['train/f1']}  |  "
            f"VAL: loss={vals['val/loss']}  acc={vals['val/acc']}  "
            f"prec={vals['val/prec']}  rec={vals['val/rec']}  f1={vals['val/f1']}"
        )

In [ ]:
# === callbacks_confusion_csvlogger.py ===
import os
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from pytorch_lightning.callbacks import Callback

class ConfusionAndLossCSVCallback(Callback):
    """
    No modifica tu módulo. Calcula pérdidas por época y guarda y_true/y_pred para reporte y CM.
    - train_loss: promedio por época desde las batches de entrenamiento
    - val_loss/val_acc: re-evalúa en validación (forward) para no depender de tus returns
    - preds.csv: con y_true/y_pred en logger_dir
    """

    def __init__(self):
        super().__init__()
        # train
        self._tr_loss_sum = 0.0
        self._tr_count = 0
        # val
        self._va_loss_sum = 0.0
        self._va_count = 0
        self._y_true = []
        self._y_pred = []

    # ---------- helpers ----------
    @staticmethod
    def _logger_dir(trainer):
        # funciona con CSVLogger y lista de loggers
        lg = getattr(trainer, "logger", None)
        if lg is None:
            return "logs"
        if hasattr(lg, "log_dir") and lg.log_dir:
            return lg.log_dir
        if isinstance(lg, (list, tuple)):
            for l in lg:
                if hasattr(l, "log_dir") and l.log_dir:
                    return l.log_dir
        return "logs"

    # ---------- TRAIN ----------
    def on_train_epoch_start(self, trainer, pl_module):
        self._tr_loss_sum = 0.0
        self._tr_count = 0

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        # outputs puede ser tensor o dict; extrae 'loss' si existe
        loss_val = None
        if torch.is_tensor(outputs):
            loss_val = outputs.detach()
        elif isinstance(outputs, dict) and "loss" in outputs:
            loss_val = outputs["loss"].detach()
        if loss_val is not None:
            self._tr_loss_sum += float(loss_val)
            self._tr_count += 1

    def on_train_epoch_end(self, trainer, pl_module):
        if self._tr_count > 0:
            tr_loss = self._tr_loss_sum / self._tr_count
            pl_module.log("train_loss", tr_loss, prog_bar=False)

    # ---------- VAL ----------
    def on_validation_epoch_start(self, trainer, pl_module):
        self._va_loss_sum = 0.0
        self._va_count = 0
        self._y_true = []
        self._y_pred = []

    def on_validation_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx=0):
        # Re-evaluamos para no depender de tu validation_step
        try:
            x, y = batch
        except Exception:
            return  # si tu batch es distinto, adapta esto
        device = next(pl_module.parameters()).device
        x = x.to(device)
        y = y.to(device)

        with torch.no_grad():
            logits = pl_module(x)
            loss = F.cross_entropy(logits, y)
            preds = logits.argmax(dim=1)

        self._va_loss_sum += float(loss.detach().cpu())
        self._va_count += 1
        self._y_true.append(y.detach().cpu().numpy())
        self._y_pred.append(preds.detach().cpu().numpy())

    def on_validation_epoch_end(self, trainer, pl_module):
        import json
        log_dir = self._logger_dir(trainer)
        os.makedirs(log_dir, exist_ok=True)

        # métricas por época
        if self._va_count > 0:
            va_loss = self._va_loss_sum / self._va_count
            y_true = np.concatenate(self._y_true) if len(self._y_true) else np.array([])
            y_pred = np.concatenate(self._y_pred) if len(self._y_pred) else np.array([])
            acc = float((y_true == y_pred).mean()) if y_true.size else 0.0

            # loggea a CSVLogger
            pl_module.log("val_loss", va_loss, prog_bar=True)
            pl_module.log("val_acc", acc, prog_bar=True)

            # guarda predicciones
            if y_true.size:
                df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred})
                # por época
                df.to_csv(os.path.join(log_dir, f"preds_epoch_{trainer.current_epoch:03d}.csv"), index=False)
                # consolidado
                df.to_csv(os.path.join(log_dir, "preds.csv"), index=False)


## Entrenamiento

In [ ]:
from pytorch_lightning.loggers import CSVLogger
from pytorch_lightning.callbacks import LearningRateMonitor
from callbacks_confusion_csvlogger import ConfusionAndLossCSVCallback

In [ ]:
def run_training():
    dm = SpectrogramDataModule(DATA_ROOT, SPLITS, BATCH_SIZE, NUM_WORKERS)
    dm.setup()
    print("Clases:", dm.ds_train.classes)
    print(f"train={len(dm.ds_train)} | val={len(dm.ds_val)} | test={len(dm.ds_test)}")

    model = CNNBiLSTMClassifier(num_classes=dm.num_classes, dropout=DROPOUT,
                                lstm_hidden=LSTM_HIDDEN, lstm_layers=LSTM_LAYERS)

    ckpt  = ModelCheckpoint(monitor='val/acc', mode='max', save_top_k=1, save_last=True,
                            filename='cnnlstm-{epoch:02d}-{val_acc:.3f}')
    early = EarlyStopping(monitor='val/acc', mode='max', patience=10)

    # crea el logger CSV (carpeta y nombre a tu gusto)
    csv_logger = CSVLogger(save_dir="logs/analysis", name="cnnlstm")

    # (opcional) monitor de LR si te sirve ver la columna lr-AdamW
    lr_cb = LearningRateMonitor(logging_interval="epoch")

    # nuestro callback nuevo
    conf_cb = ConfusionAndLossCSVCallback()
    lrmon = LearningRateMonitor(logging_interval='epoch')
    logger = CSVLogger("logs", name="cnnlstm")
    printer = EpochMetricsPrinter()

    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS,
        accelerator='auto', devices='auto',
        precision='16-mixed',
        logger=csv_logger,
        callbacks=[ckpt, early, lrmon, printer, lr_cb, conf_cb],   # <--- imprime por época
        logger=[logger],
        log_every_n_steps=25
    )

    trainer.fit(model, datamodule=dm)
    print("Best ckpt:", ckpt.best_model_path or "<none>", "| best:", ckpt.best_model_score)
    trainer.test(model=None, datamodule=dm, ckpt_path=('best' if ckpt.best_model_path else None))
    print("CSV logs:", logger.log_dir)

if __name__ == "__main__":
    run_training()

In [ ]:
# === reporte_clasificacion_y_matriz.py ===
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from pathlib import Path
import glob

# Localiza la última versión si no recuerdas el path exacto
base = Path("logs/analysis/cnnlstm")
versions = sorted(base.glob("version_*"), key=lambda p: int(p.name.split("_")[-1]))
log_dir = versions[-1] if versions else base  # o usa Path(trainer.logger.log_dir)
preds_csv = log_dir / "preds.csv"

dfp = pd.read_csv(preds_csv)
cols = {c.lower(): c for c in dfp.columns}
y_true = dfp[cols.get("y_true")]
y_pred = dfp[cols.get("y_pred")]

print(classification_report(y_true, y_pred, digits=4))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm).plot(values_format="d")
plt.title("Matriz de confusión")
plt.show()


In [ ]:
# === curvas_loss.py ===
import pandas as pd, numpy as np, re
import matplotlib.pyplot as plt
from pathlib import Path

base = Path("logs/analysis/cnnlstm")
versions = sorted(base.glob("version_*"), key=lambda p: int(p.name.split("_")[-1]))
log_dir = versions[-1] if versions else base  # o Path(trainer.logger.log_dir)
metrics_path = log_dir / "metrics.csv"

df = pd.read_csv(metrics_path)
df.columns = [re.sub(r'[\/\.]', '_', c.strip()) for c in df.columns]

epoch_col = "epoch" if "epoch" in df.columns else ("step" if "step" in df.columns else None)
if epoch_col is None:
    df["epoch"] = np.arange(len(df))
    epoch_col = "epoch"

for c in df.columns:
    if c != epoch_col:
        df[c] = pd.to_numeric(df[c], errors="coerce")

loss_cols = [c for c in df.columns
             if "loss" in c.lower()
             and not c.lower().endswith("_step")
             and c != epoch_col
             and df[c].notna().any()]

sdf = df.groupby(epoch_col, as_index=False).agg({c: "last" for c in df.columns if c != epoch_col})

if loss_cols:
    plt.figure()
    for c in loss_cols:
        plt.plot(sdf[epoch_col], sdf[c], label=c)
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("Curvas de loss"); plt.legend(); plt.show()
else:
    print("No se hallaron columnas 'loss'. Columnas disponibles:", list(df.columns))
